# Sujet 1 · Analyse de sentiments : un lexique pour lire des avis de lecteurs

**Cours « Intelligence artificielle »** · Master 2 (ANG, LGC, DID) · Université Yahia Farès, Médéa
**Mini-projet, sujet 1** (analyse de sentiments) · **Chapitre 3** (IA symbolique : des règles lisibles ; Transformers) · **Durée :** environ 1 h 30

**Ce que vous allez faire**
- réunir un petit corpus de 20 avis de lecteurs sur des romans, chacun étiqueté à la main : positif, négatif ou neutre ;
- calculer le score de chaque avis avec une méthode à lexique transparente, comme la démonstration du site : des mots notés, la négation « ne… pas / jamais », les intensifieurs « très », « vraiment » ;
- mesurer l'exactitude de la méthode par rapport à la lecture humaine, puis analyser ses erreurs : négation, ironie, mots absents du lexique.

**Mode d'emploi**
Exécutez les cellules de haut en bas : cliquez sur le bouton ▶ à gauche de chaque cellule, ou appuyez sur Maj+Entrée. Vous n'avez jamais à écrire de code. Modifiez seulement les champs des cellules intitulées « À MODIFIER ». Dans Colab, le menu « Exécution → Tout exécuter » lance tout le carnet d'un coup.

Avant chaque cellule de code, une question vous demande de **prédire** le résultat : notez votre réponse, exécutez, puis comparez avec la rubrique « Ce que vous devez voir ». Python écrit les nombres décimaux avec un point : 0.361 se lit 0,361.

> **Données fictives.** Les 20 avis de ce carnet ont été inventés pour l'exercice, et leurs étiquettes « manuelles » sont celles de l'auteur du carnet. Aucun lecteur réel n'est cité : ne présentez jamais ces chiffres comme un résultat d'enquête.

## Étape 1 · Charger les outils

Cette cellule installe **VADER** (la bibliothèque `vaderSentiment`), l'outil d'analyse de sentiments cité dans le programme. Il est conçu pour l'anglais : nous le mettrons à l'épreuve du français à l'étape 17. La cellule charge aussi pandas (tableaux), matplotlib (graphiques) et trois petits modules de Python : `io`, `math` et `re`.

**Avant d'exécuter :** cette cellule va-t-elle déjà analyser un texte, ou seulement préparer les outils ?

In [ ]:
#@title Installer VADER et charger les outils
%pip install -q vaderSentiment
import io                                  # lire un texte comme un fichier
import math                                # racine carrée
import re                                  # découper une phrase en mots
import pandas as pd                        # tableaux de données
import matplotlib.pyplot as plt            # graphiques
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyseur = SentimentIntensityAnalyzer()   # VADER, pour la comparaison de l'étape 17
pd.set_option("display.max_colwidth", 60)  # afficher les avis en entier
print("Outils chargés.")

**Ce que vous devez voir :** parfois une ligne sur l'installation, puis `Outils chargés.` Rien n'est encore analysé : la cellule prépare le travail.

**Question :** VADER est conçu pour l'anglais. Que va-t-il se passer, selon vous, s'il lit un avis écrit en français ? Notez votre réponse : vous la vérifierez à l'étape 17.

## Étape 2 · Le corpus (1/2) : dix avis ordinaires

Voici dix avis de lecteurs sur des romans, **inventés pour l'exercice**. Chaque ligne contient trois informations séparées par un point-virgule, comme dans un fichier CSV :
- le **texte** de l'avis ;
- l'étiquette **manuelle** : la lecture humaine (positif, négatif ou neutre). C'est la référence à laquelle on comparera la machine ;
- la **difficulté** prévue pour la machine : `simple` ou `négation`.

pandas lit ce texte comme s'il s'agissait d'un fichier et en fait un tableau.

**Avant d'exécuter :** lisez les dix avis dans la cellule. Êtes-vous d'accord avec chaque étiquette ? Lequel vous semble le plus difficile pour une machine ?

In [ ]:
#@title Le corpus (1/2) : dix avis « ordinaires » (données fictives)
# Une ligne par avis : texte ; étiquette manuelle ; difficulté prévue
avis_ordinaires = """Un roman magnifique, je l'ai adoré.;positif;simple
Une histoire captivante, très bien écrite.;positif;simple
Livre ennuyeux et beaucoup trop long.;négatif;simple
Style lourd, intrigue faible : quelle déception.;négatif;simple
Le roman compte 300 pages et se passe à Paris.;neutre;simple
Lu pour mon cours de littérature comparée.;neutre;simple
Je n'ai pas aimé ce livre.;négatif;négation
Pas mal du tout, je le recommande.;positif;négation
Je ne l'ai jamais trouvé intéressant.;négatif;négation
Je ne peux pas dire que ce livre soit bon.;négatif;négation"""
colonnes = ["texte", "manuel", "difficulte"]
ordinaires = pd.read_csv(io.StringIO(avis_ordinaires), sep=";", names=colonnes)
ordinaires

**Ce que vous devez voir :** un tableau de 10 lignes numérotées de 0 à 9 (Python compte à partir de 0) et 3 colonnes : `texte`, `manuel`, `difficulte`. On y trouve 3 avis positifs, 5 négatifs et 2 neutres.

**Question :** « Lu pour mon cours de littérature comparée. » est-il vraiment neutre ? Qu'est-ce qui décide de votre étiquette : les mots de l'avis, ou ce que vous imaginez de son auteur ?

## Étape 3 · Le corpus (2/2) : dix avis pièges

Dix autres avis, eux aussi inventés, contiennent des pièges bien connus de l'analyse de sentiments :
- l'**ironie** : des mots positifs pour dire le contraire ;
- des **mots absents** du lexique que nous utiliserons (« haletant », « dévoré », une émoticône) ;
- un **thème triste** (la guerre, la mort) qui n'est pas forcément un jugement négatif sur le livre.

La cellule réunit les 20 avis dans un seul tableau, `df`, puis compte les étiquettes manuelles.

**Avant d'exécuter :** sur les 20 avis, combien de positifs, de négatifs et de neutres ?

In [ ]:
#@title Le corpus (2/2) : dix avis « pièges » (données fictives)
# Pièges : ironie, mot absent du lexique, thème triste
avis_pieges = """Génial, encore 900 pages où il ne se passe rien.;négatif;ironie
Bravo à l'auteur : je me suis endormi page 10.;négatif;ironie
Quel plaisir de relire dix fois la même scène.;négatif;ironie
Impossible à lâcher, je l'ai dévoré en une nuit !;positif;mot absent
Intrigue prévisible, personnages en carton.;négatif;mot absent
Une plume ciselée, des chapitres haletants.;positif;mot absent
Fini cette nuit :);positif;mot absent
Guerre, peur et mort : un roman sublime.;positif;thème triste
Le roman raconte la mort du père à la guerre.;neutre;thème triste
Une histoire triste, mais tellement belle.;positif;thème triste"""
pieges = pd.read_csv(io.StringIO(avis_pieges), sep=";", names=colonnes)
df = pd.concat([ordinaires, pieges], ignore_index=True)   # les 20 avis, de 0 à 19
df["manuel"].value_counts()

**Ce que vous devez voir :** le compte des étiquettes manuelles : `négatif 9`, `positif 8`, `neutre 3` (20 avis en tout).

**Question :** les avis neutres sont rares ici (3 sur 20). Est-ce aussi le cas dans les avis que vous lisez en ligne ? Pourquoi un lecteur écrit-il rarement un avis neutre ?

## Étape 4 · Le lexique : des mots et leurs scores

Une méthode à lexique ne comprend pas les phrases : elle cherche chaque mot dans une liste de **mots notés**, de −3 (très négatif) à +3 (très positif). Notre lexique reprend une partie de celui de la démonstration du site (« Lire une phrase comme une machine »), avec les mêmes scores et quelques formes ajoutées comme « aimé » : les mots positifs d'abord, puis les négatifs. Deux petites listes s'y ajoutent :
- les **négations** (`ne`, `n'`, `pas`, `jamais`, `rien`…), qui retourneront le mot qui les suit ;
- les **intensifieurs** (`très`, `vraiment`, `trop`, `tellement`), qui le renforceront.

C'est de l'**IA symbolique** (chapitre 3) : des connaissances écrites à la main, lisibles par un humain. VADER, l'outil anglais du programme, fonctionne de la même façon, avec un lexique bien plus grand.

**Avant d'exécuter :** parcourez le lexique, puis les 20 avis. Citez trois mots des avis qui manquent au lexique.

In [ ]:
#@title Notre lexique : des mots et leurs scores (de -3 à +3)
LEXIQUE = {
    "beau": 2, "belle": 2, "magnifique": 3, "sublime": 3, "superbe": 3, "génial": 3,
    "bon": 1.5, "bonne": 1.5, "bien": 1.5, "aime": 2, "aimé": 2, "adore": 3, "adoré": 3,
    "captivant": 2.5, "captivante": 2.5, "émouvant": 2, "émouvante": 2, "touchant": 2,
    "intéressant": 1.5, "plaisir": 2, "bravo": 2.5, "réussi": 2, "excellent": 3,
    "mauvais": -2, "mal": -1.5, "ennuyeux": -2, "ennuyeuse": -2, "décevant": -2,
    "décevante": -2, "déçu": -2, "déception": -2, "lourd": -1, "faible": -1, "raté": -2,
    "horrible": -3, "triste": -2, "peur": -2, "mort": -2.5, "guerre": -2.5, "larmes": -2,
}
NEGATIONS = ["ne", "n'", "pas", "jamais", "rien", "aucun", "sans"]
INTENSIFS = ["très", "vraiment", "trop", "tellement"]
print("Mots dans le lexique :", len(LEXIQUE))
print("Négations :", len(NEGATIONS), "| intensifieurs :", len(INTENSIFS))

**Ce que vous devez voir :** `Mots dans le lexique : 40`, puis `Négations : 7 | intensifieurs : 4`. Parmi les mots des avis absents du lexique : « roman », « lâcher », « dévoré », « prévisible », « ciselée », « haletants », « endormi »…

**Question :** 40 mots suffisent-ils pour lire des avis de lecteurs ? Quels mots de la critique littéraire (ou de vos cours de didactique) faudrait-il ajouter en priorité ?

## Étape 5 · Chercher un mot dans le lexique

Cette cellule affiche le score de quelques mots, puis celui du mot que vous tapez dans le champ `mot` (à droite de la cellule dans Colab). « absent du lexique » veut dire que la méthode ignorera ce mot : il comptera pour zéro.

**Avant d'exécuter :** le mot « roman » a-t-il un score ? Et l'émoticône « :) » ?

In [ ]:
#@title À MODIFIER : chercher un mot dans le lexique
mot = "sublime" #@param {type:"string"}
for m in ["bon", "ennuyeux", "roman", ":)", mot]:
    print(m, "->", LEXIQUE.get(m.lower(), "absent du lexique"))

**Ce que vous devez voir :** `bon -> 1.5`, `ennuyeux -> -2`, `roman -> absent du lexique`, `:) -> absent du lexique`, `sublime -> 3`.

**Question :** tapez dans le champ « haletant », puis « chef-d'œuvre », puis « bof ». Un mot absent compte pour zéro : que va obtenir un avis écrit uniquement avec des mots que le lexique ne connaît pas ?

## Étape 6 · Découper une phrase en mots

Pour chercher les mots, il faut d'abord découper la phrase : tout passe en minuscules, l'apostrophe est détachée (« n'ai » devient « n' » + « ai »), la ponctuation disparaît, mais les émoticônes `:)` et `:(` sont gardées. La fonction `mots_connus` renvoie les mots de la phrase présents dans le lexique, avec leur score.

**Avant d'exécuter :** en combien de morceaux la phrase « Je n'ai pas aimé ce livre :( » sera-t-elle découpée ? Lesquels sont dans le lexique ?

In [ ]:
#@title Découper une phrase en mots, repérer les mots connus du lexique
def decouper(phrase):
    # minuscules, apostrophe détachée (« n'ai » -> « n' » + « ai »), émoticônes :) et :( gardées
    phrase = phrase.lower().replace("’", "'").replace("'", "' ")
    return re.findall(r":\)|:\(|[\w'-]+", phrase)

def mots_connus(phrase):
    trouves = []
    for mot in decouper(phrase):
        if mot in LEXIQUE:
            trouves.append((mot, LEXIQUE[mot]))   # (mot, score)
    return trouves

print(decouper("Je n'ai pas aimé ce livre :("))
print(mots_connus("Je n'ai pas aimé ce livre :("))

**Ce que vous devez voir :** `['je', "n'", 'ai', 'pas', 'aimé', 'ce', 'livre', ':(']`, 8 morceaux, puis `[('aimé', 2)]` : un seul mot connu. L'émoticône `:(` est gardée, mais notre lexique ne la connaît pas.

**Question :** « aimé » est reconnu avec un score positif (+2), alors que l'avis est négatif. Quel mot de la phrase devra retourner ce score ? Est-il juste avant « aimé » ?

## Étape 7 · La méthode : le score d'une phrase

Voici toute la méthode, en quatre règles :
1. on additionne les scores des mots connus du lexique ;
2. si le mot **juste avant** est un intensifieur (« très bon »), le score du mot est multiplié par 1,3 : 30 % plus fort ;
3. si l'un des **trois mots qui précèdent** est une négation (« pas bon », « n'ai pas aimé »), le score est retourné et affaibli : multiplié par −0,74, le chiffre de VADER ;
4. le total est ramené entre −1 et +1 par la formule de VADER : total / √(total² + 15).

**Avant d'exécuter :** dans « Ce livre est bon. », seul « bon » est connu (+1,5). Le score final sera-t-il plus grand ou plus petit que 1,5 ?

In [ ]:
#@title La méthode : calculer le score d'une phrase (de -1 à +1)
def score(phrase):
    mots = decouper(phrase)
    total = 0
    for i, mot in enumerate(mots):                 # i = position du mot dans la phrase
        if mot in LEXIQUE:
            valeur = LEXIQUE[mot]
            avant = mots[max(0, i - 3):i]          # les 3 mots qui précèdent
            if len(avant) > 0 and avant[-1] in INTENSIFS:
                valeur = valeur * 1.3              # « très bon » : 30 % plus fort
            if any(m in NEGATIONS for m in avant):
                valeur = valeur * -0.74            # « pas bon » : retourné et affaibli
            total = total + valeur
    return round(total / math.sqrt(total * total + 15), 3)   # ramené entre -1 et +1
print(score("Ce livre est bon."))

**Ce que vous devez voir :** `0.361`. Le calcul : 1,5 / √(1,5² + 15) = 1,5 / 4,15 ≈ 0,361. Le score est plus petit que 1,5 : la formule écrase tous les totaux entre −1 et +1.

**Question :** avec cette formule, quel total faut-il pour atteindre un score de 0,9 ? Un seul mot très positif suffit-il ? (Essayez à l'étape 9 avec plusieurs mots positifs.)

## Étape 8 · Négation et intensifieurs : quatre variantes

La même phrase en quatre versions : neutre, renforcée par « très », niée par « ne… pas », puis les deux à la fois.

**Avant d'exécuter :** classez les quatre phrases de la plus négative à la plus positive. « pas très bon » sera-t-il plus ou moins négatif que « pas bon » ?

In [ ]:
#@title Négation et intensifieurs : quatre variantes d'une phrase
variantes = ["Ce livre est bon.", "Ce livre est très bon.",
             "Ce livre n'est pas bon.", "Ce livre n'est pas très bon."]
for phrase in variantes:
    print(score(phrase), "|", phrase)

**Ce que vous devez voir :** `0.361` (bon), `0.45` (très bon), `-0.276` (pas bon), `-0.349` (pas très bon). La négation retourne bien le score, et « très » le renforce, dans les deux sens.

**Question :** pour la méthode, « pas très bon » est plus négatif que « pas bon ». Est-ce votre lecture ? En français, « pas très bon » est souvent une litote, plus polie. Qu'est-ce qu'une règle aussi simple ne peut pas saisir ?

## Étape 9 · À vous : votre phrase

Tapez votre propre phrase dans le champ `phrase`, puis exécutez. La cellule affiche le découpage, les mots connus et le score. La phrase proposée est celle de la démonstration du site.

**Avant d'exécuter :** « magnifique » vaut +3, « décevante » −2, renforcé par « vraiment ». Le score sera-t-il positif ou négatif ?

In [ ]:
#@title À MODIFIER : votre phrase
phrase = "Ce roman est magnifique, mais la fin est vraiment décevante." #@param {type:"string"}
print("Mots :", decouper(phrase))
print("Mots connus :", mots_connus(phrase))
print("Score :", score(phrase))

**Ce que vous devez voir :** `Mots connus : [('magnifique', 3), ('décevante', -2)]` et `Score : 0.103`, légèrement positif : 3 + (−2 × 1,3) = 0,4, ramené à 0,103.

La démonstration du site classe pourtant cette phrase comme **négative** : elle applique une règle de plus, qui donne plus de poids à ce qui suit « mais » (comme VADER avec *but*). Notre méthode, plus simple, n'a pas cette règle.

**Question :** laquelle des deux lectures est la bonne pour vous ? Essayez ensuite une phrase ironique, une litote (« ce n'est pas mauvais »), une phrase en darja : que devient le score ?

## Étape 10 · Le score des 20 avis

On applique la méthode aux 20 avis, puis on transforme chaque score en étiquette grâce à un **seuil** : au-dessus de +seuil, positif ; en dessous de −seuil, négatif ; entre les deux, neutre. Les auteurs de VADER recommandent 0,05 ; le curseur `seuil` permet de le changer.

**Avant d'exécuter :** combien d'avis l'outil va-t-il classer comme neutres ? Pensez aux avis dont aucun mot n'est dans le lexique.

In [ ]:
#@title À MODIFIER : le seuil de neutralité, puis le score de chaque avis
seuil = 0.05 #@param {type:"slider", min:0, max:0.5, step:0.05}
def etiqueter(s):
    # au-dessus du seuil : positif ; en dessous de -seuil : négatif ; sinon : neutre
    if s >= seuil:
        return "positif"
    if s <= -seuil:
        return "négatif"
    return "neutre"

df["score"] = df["texte"].apply(score)
df["outil"] = df["score"].apply(etiqueter)
df

**Ce que vous devez voir :** le tableau des 20 avis avec deux colonnes de plus, `score` et `outil`. Le plus positif est l'avis 0 (0.84), le plus négatif l'avis 18 (−0.791, « Le roman raconte la mort du père à la guerre. »). 6 avis ont un score de 0.0 et sont classés neutres : les avis 4 et 5, et les quatre avis « mot absent » (13 à 16). L'avis ironique 10 (« Génial, encore 900 pages… ») obtient 0.612 : positif.

**Question :** trouvez dans le tableau un avis où l'outil a raison pour une mauvaise raison, ou par hasard. (Indice : regardez les avis neutres.)

## Étape 11 · Diagramme 1 : la répartition des sentiments

Le diagramme demandé par le mini-projet : combien d'avis positifs, neutres et négatifs, selon votre lecture (violet) et selon l'outil (orange). Le tableau des chiffres s'affiche au-dessus.

**Avant d'exécuter :** l'outil trouvera-t-il plus, moins, ou autant d'avis positifs que vous ?

In [ ]:
#@title Diagramme 1 : la répartition des sentiments, selon vous et selon l'outil
ordre = ["négatif", "neutre", "positif"]
comptes = pd.DataFrame({"votre lecture": df["manuel"].value_counts(),
                        "l'outil": df["outil"].value_counts()})
comptes = comptes.reindex(ordre).fillna(0).astype(int)   # même ordre ; 0 si absent
print(comptes)
comptes.plot(kind="bar", rot=0, color=["#4a3aa7", "#eb6834"], figsize=(6, 3.5))
plt.title("Répartition des sentiments (20 avis fictifs)")
plt.ylabel("nombre d'avis")
plt.xlabel("")
plt.show()

**Ce que vous devez voir :** vous : 9 négatifs, 3 neutres, 8 positifs ; l'outil : 6 négatifs, 6 neutres, 8 positifs. L'outil trouve exactement autant d'avis positifs que vous… mais ce ne sont pas les mêmes : l'étape 13 le montrera.

**Question :** un rapport qui ne montrerait que ce diagramme pourrait-il conclure que l'outil « lit bien » les avis positifs ? Pourquoi faut-il comparer avis par avis ?

## Étape 12 · Diagramme 2 : le score de chaque avis

Une barre par avis : vers la droite si le score est positif, vers la gauche s'il est négatif. La couleur donne **votre** étiquette : bleu = positif, gris = neutre, rouge = négatif. Les pointillés marquent le seuil.

**Avant d'exécuter :** une barre bleue tournée vers la gauche, qu'est-ce que cela signifie ?

In [ ]:
#@title Diagramme 2 : le score de chaque avis, coloré selon votre lecture
couleurs = {"positif": "#2a78d6", "neutre": "#8f8d87", "négatif": "#e34948"}
fig, ax = plt.subplots(figsize=(8, 7))
for etiquette, couleur in couleurs.items():
    partie = df[df["manuel"] == etiquette]
    ax.barh(partie.index, partie["score"], color=couleur, label="lu comme " + etiquette)
ax.set_yticks(df.index, df.index.astype(str) + ". " + df["texte"].str[:34])
ax.invert_yaxis()                                   # l'avis 0 en haut
ax.axvline(0, color="black", linewidth=0.8)
ax.vlines([-seuil, seuil], -0.5, len(df) - 0.5, colors="gray", linestyles="dotted")
ax.set_xlim(-1, 1)
ax.set_xlabel("score de l'outil (de -1 à +1)")
ax.set_title("Score de chaque avis (données fictives)")
ax.legend(loc="center left")
plt.show()

**Ce que vous devez voir :** 20 barres. En haut, les avis simples vont du bon côté. Plus bas, les trois avis ironiques (10 à 12) sont rouges mais tournés vers la droite ; les avis 17 et 18 (guerre, mort) partent loin vers la gauche alors que 17 est bleu et 18 gris ; les avis 4, 5 et 13 à 16 n'ont pas de barre du tout (score 0).

**Question :** une barre bleue tournée vers la gauche, ou rouge vers la droite, est une erreur de l'outil. Combien en comptez-vous ? Et les avis sans barre, sont-ils des erreurs ?

## Étape 13 · Exactitude et tableau de confusion

L'**exactitude** (*accuracy*) est la part des avis où l'outil donne la même étiquette que vous. Le **tableau de confusion** croise les deux lectures : une ligne par étiquette manuelle, une colonne par étiquette de l'outil. Les bonnes réponses sont sur la diagonale.

**Avant d'exécuter :** d'après les diagrammes, l'exactitude sera-t-elle plutôt de 30 %, 50 % ou 80 % ?

In [ ]:
#@title Exactitude : l'outil est-il d'accord avec votre lecture ?
df["accord"] = df["manuel"] == df["outil"]   # True si l'outil donne la même étiquette que vous
print("Avis bien classés :", df["accord"].sum(), "sur", len(df))
print("Exactitude :", round(100 * df["accord"].mean()), "%")
# Tableau de confusion : lignes = votre étiquette, colonnes = celle de l'outil
pd.crosstab(df["manuel"], df["outil"])

**Ce que vous devez voir :** `Avis bien classés : 10 sur 20` et `Exactitude : 50 %`. Sur la diagonale du tableau : 4 négatifs, 2 neutres, 4 positifs bien classés. Parmi les 9 avis négatifs, 4 sont classés positifs ; parmi les 8 positifs, 3 sont classés neutres.

Attention : ce corpus a été construit **pour piéger** la méthode (la moitié des avis sont des pièges). Sur des avis ordinaires, l'exactitude serait plus haute ; sur votre corpus réel, il faudra la mesurer.

**Question :** un outil qui répondrait toujours « négatif » aurait 9 bonnes réponses sur 20 (45 %). Notre méthode fait-elle beaucoup mieux ? Que faut-il toujours donner à côté d'une exactitude ?

## Étape 14 · L'exactitude selon la difficulté

On regroupe les avis selon la difficulté prévue (colonne `difficulte`) et on calcule l'exactitude de chaque groupe, en pourcentage.

**Avant d'exécuter :** quel groupe aura la meilleure exactitude ? Lequel aura 0 % ?

In [ ]:
#@title Exactitude selon la difficulté prévue (en %)
par_difficulte = df.groupby("difficulte", sort=False)["accord"].mean() * 100
par_difficulte.round()

**Ce que vous devez voir :** `simple 100.0`, `négation 75.0`, `ironie 0.0`, `mot absent 0.0`, `thème triste 33.0` (en %). Sur les 10 avis ordinaires, la méthode a 9 bonnes réponses ; sur les 10 avis pièges, une seule.

**Question :** l'exactitude globale (50 %) cachait ces écarts. Si votre corpus réel contient beaucoup d'ironie (des commentaires sur une adaptation ratée, par exemple), à quoi pouvez-vous vous attendre ?

## Étape 15 · Analyse des erreurs

Chiffrer ne suffit pas : il faut relire les erreurs. Pour chaque avis mal classé, la cellule affiche sa difficulté, votre étiquette, celle de l'outil, et les mots que le lexique a reconnus.

**Avant d'exécuter :** pour l'avis « Je ne peux pas dire que ce livre soit bon. », pourquoi la négation n'a-t-elle pas fonctionné ? (Comptez les mots entre « pas » et « bon ».)

In [ ]:
#@title Analyse des erreurs : relire chaque avis mal classé
erreurs = df[df["accord"] == False]
print(len(erreurs), "avis mal classés\n")
for i, ligne in erreurs.iterrows():
    print(i, "[" + str(ligne["difficulte"]) + "]", ligne["texte"])
    print("   vous :", ligne["manuel"], "| outil :", ligne["outil"], "| score :", ligne["score"])
    print("   mots connus :", mots_connus(ligne["texte"]))

**Ce que vous devez voir :** 10 avis mal classés. On peut les ranger en quatre familles :
- **négation éloignée** (avis 9) : « pas » est à 5 mots de « bon » ; la règle ne regarde que les 3 mots qui précèdent, donc « bon » reste positif (0.361) ;
- **ironie** (avis 10, 11, 12) : « génial », « bravo », « plaisir » sont positifs, mais le lecteur veut dire le contraire. Le sens vient du reste de la phrase (900 pages où rien ne se passe, s'endormir page 10) et de ce que l'on sait des lecteurs : aucun lexique n'y a accès ;
- **mots absents du lexique** (avis 13 à 16) : aucun mot connu, score 0, donc « neutre ». Le neutre de l'outil est ici de l'ignorance, pas de la neutralité ;
- **thème triste** (avis 17 et 18) : « guerre », « peur », « mort » décrivent l'histoire, pas le jugement du lecteur. « un roman sublime » (+3) ne pèse pas assez face à trois mots négatifs.

**Question :** pour chaque famille, proposez une amélioration de la méthode (une règle, des mots à ajouter, une autre fenêtre…). Laquelle vous paraît impossible à régler avec un lexique ?

## Étape 16 · Ajouter un mot au lexique

Le plus simple à corriger : les mots absents. Tapez un mot dans `mot_nouveau`, choisissez son score avec le curseur, puis exécutez : le mot entre dans le lexique et les 20 avis sont recalculés. Pour repartir du lexique d'origine, relancez l'étape 4.

**Avant d'exécuter :** avec « dévoré » à +2,5, combien d'avis seront bien classés ?

In [ ]:
#@title À MODIFIER : ajouter un mot au lexique, puis tout recalculer
mot_nouveau = "dévoré" #@param {type:"string"}
score_nouveau = 2.5 #@param {type:"slider", min:-3, max:3, step:0.5}
LEXIQUE[mot_nouveau.lower()] = score_nouveau
df["score"] = df["texte"].apply(score)
df["outil"] = df["score"].apply(etiqueter)
df["accord"] = df["manuel"] == df["outil"]
print("Mots dans le lexique :", len(LEXIQUE))
print("Avis bien classés :", df["accord"].sum(), "sur", len(df))
print("Exactitude :", round(100 * df["accord"].mean()), "%")

**Ce que vous devez voir :** `Mots dans le lexique : 41`, `Avis bien classés : 11 sur 20`, `Exactitude : 55 %` : l'avis 13 (« Impossible à lâcher, je l'ai dévoré en une nuit ! ») est désormais positif. En ajoutant ensuite « haletants » (+2) puis « prévisible » (−1,5), on monte à 12, puis 13 avis bien classés (65 %).

**Question :** en ajoutant les mots qui manquent à **ces** 20 avis, on améliore le score… sur ces 20 avis. Est-ce une amélioration honnête ? Que se passera-t-il sur de nouveaux avis ? (C'est le problème du surapprentissage, chapitre 3.)

## Étape 17 · Et VADER, sur du français ?

Pourquoi ne pas utiliser directement VADER, l'outil du programme ? La cellule lui fait lire trois avis en anglais, puis leur traduction française, et affiche à côté le score de notre méthode.

**Avant d'exécuter :** reprenez votre prédiction de l'étape 1. Quels scores VADER donnera-t-il aux phrases françaises ?

In [ ]:
#@title VADER lit trois avis en anglais, puis leur traduction française
paires = [("A magnificent book, I adored it.", "Un roman magnifique, je l'ai adoré."),
          ("Boring book, and far too long.", "Livre ennuyeux et beaucoup trop long."),
          ("I did not like this book.", "Je n'ai pas aimé ce livre.")]
for anglais, francais in paires:
    print("VADER :", analyseur.polarity_scores(anglais)["compound"], "|", anglais)
    print("VADER :", analyseur.polarity_scores(francais)["compound"], "|", francais,
          "| notre méthode :", score(francais))
    print()

**Ce que vous devez voir :** en anglais, VADER donne 0.7717, −0.3182 et −0.2755 ; en français, 0.0 pour les trois phrases : il ne reconnaît aucun mot. Notre petit lexique donne 0.84, −0.459 et −0.357.

**Question :** le cours propose trois solutions pour le français : un modèle multilingue, TextBlob avec l'extension française, ou la traduction vers l'anglais avant VADER. Quel risque la traduction fait-elle courir à la tonalité d'un avis ?

## Étape 18 (facultative) · Un Transformer multilingue

> **Cellule en ligne.** Elle a besoin d'une connexion Internet ; elle fonctionne dans Colab. Elle télécharge un modèle de plusieurs centaines de Mo (une à deux minutes la première fois). Aucune autre cellule n'en dépend : vous pouvez la sauter.

Le modèle `nlptown/bert-base-multilingual-uncased-sentiment` est un Transformer de type BERT (chapitre 3), affiné sur des avis de produits en six langues, dont le français et l'anglais (pas l'arabe). Il ne consulte aucun lexique : il a **appris** à partir d'exemples annotés, et donne à chaque avis une note de 1 à 5 étoiles. La cellule traduit les étoiles en étiquettes (1-2 = négatif, 3 = neutre, 4-5 = positif) et calcule son exactitude.

**Avant d'exécuter :** le Transformer fera-t-il mieux que les 50 % de notre lexique ? Sur quels pièges ?

In [ ]:
#@title (En ligne) Un Transformer multilingue note chaque avis de 1 à 5 étoiles
from transformers import pipeline
modele = "nlptown/bert-base-multilingual-uncased-sentiment"   # modèle du Hub de Hugging Face
classifieur = pipeline("sentiment-analysis", model=modele)
etoiles = []
for texte in df["texte"]:
    resultat = classifieur(texte)[0]            # par exemple {'label': '4 stars', 'score': 0.52}
    etoiles.append(int(resultat["label"][0]))   # « 4 stars » -> 4
avis_bert = pd.DataFrame({"texte": df["texte"], "manuel": df["manuel"], "etoiles": etoiles})
vers_etiquette = {1: "négatif", 2: "négatif", 3: "neutre", 4: "positif", 5: "positif"}
avis_bert["modele"] = avis_bert["etoiles"].map(vers_etiquette)   # 1-2 étoiles = négatif...
exactitude = (avis_bert["modele"] == avis_bert["manuel"]).mean()
print("Exactitude du Transformer :", round(100 * exactitude), "%")
avis_bert

**Ce que vous devez voir :** après le téléchargement, `Exactitude du Transformer : … %`, puis le tableau des 20 avis avec leur nombre d'étoiles. Les chiffres exacts peuvent varier d'une version à l'autre : notez les vôtres. Regardez surtout les avis ironiques (10 à 12) et les avis « mot absent » (13 à 16).

**Question :** le Transformer est plus exact sur certains pièges, mais il ne peut pas **expliquer** ses notes, alors que notre méthode montre chaque mot et chaque règle. Pour un mémoire, que préférez-vous : un outil plus exact, ou un outil transparent ? Pourquoi ?

## À vous

Modifiez seulement les champs, puis relancez les cellules à partir de celle que vous avez changée (dans Colab : « Exécution → Exécuter les cellules suivantes »).
1. **Étape 10 :** mettez le seuil à 0,3, puis relancez les étapes 10 à 15. Les avis 7, 8 et 19 deviennent neutres : pourquoi ? L'exactitude baisse : dans quel cas un seuil plus prudent serait-il tout de même utile ?
2. **Étape 16 :** ajoutez un à un « haletants » (+2), « prévisible » (−1,5), « carton » (−1). Jusqu'où monte l'exactitude ? Ajoutez enfin « roman » avec le score +1 : que devient l'avis 4 ? (VADER a exactement ce problème avec le mot anglais *novel*.)
3. **Étape 9 :** écrivez un avis ironique qui trompe la méthode, puis un avis clairement positif sans aucun mot du lexique. Ajoutez-les à votre liste d'exemples pour le rapport.

## Pour le mini-projet : vos propres données

Pour votre mini-projet, remplacez les avis fictifs par **votre** corpus :
- préparez un fichier CSV (dans un tableur : « Enregistrer sous… → CSV UTF-8 ») avec une colonne `texte` (un texte par ligne) et une colonne `manuel` : votre annotation, écrite exactement `positif`, `négatif` ou `neutre`. Une colonne `difficulte` est facultative. Le séparateur peut être la virgule ou le point-virgule ;
- des textes publics, collectés en respectant les conditions d'utilisation du site, sans noms d'utilisateurs (en Algérie, la loi n° 18-07 du 10 juin 2018 encadre les données personnelles) ; gardez à part la source et la date de chaque texte ;
- cochez la case ci-dessous et exécutez la cellule : dans Colab, une fenêtre vous demande le fichier ; dans Jupyter, placez le fichier à côté du carnet sous le nom `mon_corpus.csv` ;
- relancez ensuite les étapes 10 à 16 : scores, diagrammes, exactitude, erreurs.

**Avant d'exécuter :** si vous laissez la case décochée, que va faire la cellule ?

In [ ]:
#@title À MODIFIER : charger votre propre corpus (fichier CSV)
charger_mon_fichier = False #@param {type:"boolean"}
if charger_mon_fichier:
    try:
        from google.colab import files        # n'existe que dans Colab
        envoi = files.upload()                # fenêtre pour choisir le fichier
        nom_fichier = list(envoi.keys())[0]
    except ImportError:                       # Jupyter : fichier placé à côté du carnet
        nom_fichier = "mon_corpus.csv"
    df = pd.read_csv(nom_fichier, sep=None, engine="python")   # devine , ou ;
    if "difficulte" not in df.columns:
        df["difficulte"] = "non précisée"
    print(len(df), "textes chargés ; colonnes :", list(df.columns))
else:
    print("Case décochée : le carnet garde les avis fictifs.")

**Ce que vous devez voir :** avec la case décochée, `Case décochée : le carnet garde les avis fictifs.` Avec votre fichier, le nombre de textes chargés et la liste de vos colonnes, qui doit contenir `texte` et `manuel`.

**Question :** le cours demande d'annoter à la main un échantillon d'une cinquantaine de textes. Pourquoi ne pas annoter tout le corpus ? Et pourquoi ne pas se passer d'annotation ?

## À rendre

Répondez par écrit, en français, en quelques lignes pour chaque question.
1. **Mesurer.** Donnez l'exactitude de la méthode sur le corpus et selon la difficulté (étapes 13 et 14). Pourquoi l'exactitude globale ne suffit-elle pas à juger l'outil ?
2. **Expliquer les erreurs.** Choisissez trois avis mal classés de familles différentes (négation, ironie, mot absent, thème triste). Pour chacun, expliquez l'erreur à partir des mots connus et des règles de la méthode.
3. **Pour votre discipline.** « Un score mesure la couleur des mots employés, pas la qualité d'un jugement critique » (cours, mini-projet 1). Discutez cette phrase à partir d'un avis du corpus, puis dites comment vous vérifieriez l'outil sur votre propre corpus : réception d'un roman (LGC), commentaires d'apprenants ou retours sur un cours (DID).

## Ce qu'il faut retenir

- Une **méthode à lexique** additionne les scores de mots notés, avec quelques règles (négation, intensifieurs, seuil) : chaque score s'explique, c'est de l'IA symbolique transparente.
- On évalue toujours l'outil contre une **annotation manuelle** : exactitude, tableau de confusion, puis relecture des erreurs. Une même répartition globale peut cacher beaucoup d'erreurs.
- Ses limites sont connues : **ironie**, **négation éloignée**, **mots absents** du lexique, **thème** confondu avec le jugement, langue non prévue (VADER ne lit pas le français).

**Leçons du site :** [Mini-projet, sujet 1 : analyse de sentiments](https://progremer04.github.io/canva-au-for-ang-/#mp-sujet-1) · [Chapitre 3 : l'IA symbolique](https://progremer04.github.io/canva-au-for-ang-/#c3-symbolique) · [Chapitre 3 : l'apprentissage profond et les Transformers](https://progremer04.github.io/canva-au-for-ang-/#c3-deep)